In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Creează un pass manager pentru decuplarea dinamică

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Îți recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Această pagină demonstrează cum să folosești pass-ul [`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) pentru a adăuga o tehnică de suprimare a erorilor numită _decuplare dinamică_ (dynamical decoupling) în Circuit.

Decuplarea dinamică funcționează prin adăugarea de secvențe de impulsuri (cunoscute ca _secvențe de decuplare dinamică_) la Qubiți inactivi, pentru a-i roti în jurul sferei Bloch, ceea ce anulează efectul canalelor de zgomot, suprimând astfel decoerența. Aceste secvențe de impulsuri sunt similare cu impulsurile de refocalizare utilizate în rezonanța magnetică nucleară. Pentru o descriere completă, consultă [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560).

Deoarece pass-ul `PadDynamicalDecoupling` operează doar pe circuite planificate și implică Gate-uri care nu sunt neapărat Gate-uri de bază ale țintei noastre, vei avea nevoie și de pass-urile [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) și [`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

Obține informațiile despre `target` din `backend` și salvează numele operațiunilor ca `basis_gates`, deoarece `target`-ul va trebui modificat pentru a adăuga informații de sincronizare pentru Gate-urile utilizate în decuplarea dinamică.

> **Note:** Backend-ul mock `FakeSherbrooke` din `qiskit_ibm_runtime` este folosit în aceste exemple, dar poți încerca pe orice backend real sau fals compatibil cu Qiskit. Rezultatele tale ar putea fi diferite.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

target = backend.target
basis_gates = list(target.operation_names)

Creează un Circuit `efficient_su2` ca exemplu. Mai întâi, transpilează Circuit-ul pentru Backend, deoarece impulsurile de decuplare dinamică trebuie adăugate după ce Circuit-ul a fost transpilat și planificat. Decuplarea dinamică funcționează cel mai bine atunci când există mult timp inactiv în Circuit-urile cuantice — adică există Qubiți care nu sunt folosiți în timp ce alții sunt activi. Acesta este cazul în acest Circuit, deoarece Gate-urile `ecr` cu doi Qubiți sunt aplicate secvențial în acest ansatz.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg)

O secvență de decuplare dinamică este o serie de Gate-uri care compun identitatea și sunt espaciate regulat în timp. De exemplu, începe prin a crea o secvență simplă numită XY4, formată din patru Gate-uri.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

Datorită sincronizării regulate a secvențelor de decuplare dinamică, informațiile despre `YGate` trebuie adăugate la `target`, deoarece acesta *nu* este un Gate de bază, spre deosebire de `XGate`. Știm *a priori* că `YGate` are aceeași durată și eroare ca `XGate`, deci putem prelua acele proprietăți din `target` și le putem adăuga înapoi pentru obiectele `YGate`. De aceea `basis_gates` a fost salvat separat, deoarece adăugăm instrucțiunea `YGate` la `target`, deși nu este un Gate de bază real al lui `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

Circuit-urile ansatz precum `efficient_su2` sunt parametrizate, deci trebuie să li se asocieze valori înainte de a fi trimise la Backend. Aici, atribuie parametri aleatori.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

În continuare, execută pass-urile personalizate. Instanțiază `PassManager`-ul cu `ALAPScheduleAnalysis` și `PadDynamicalDecoupling`. Rulează mai întâi `ALAPScheduleAnalysis` pentru a adăuga informații de sincronizare despre Circuit-ul cuantic, înainte ca secvențele de decuplare dinamică espaciate regulat să poată fi adăugate. Aceste pass-uri sunt rulate pe Circuit cu `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

Folosește instrumentul de vizualizare [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) pentru a vedea sincronizarea Circuit-ului și pentru a confirma că o secvență espaciată regulat de obiecte `XGate` și `YGate` apare în Circuit.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg)

În final, deoarece `YGate` nu este un Gate de bază real al Backend-ului nostru, aplică manual pass-ul `BasisTranslator` (acesta este un pass implicit, dar este executat înainte de planificare, deci trebuie aplicat din nou). Biblioteca de echivalențe de sesiune este o bibliotecă de echivalențe de Circuit-uri care permite Transpiler-ului să descompună Circuit-urile în Gate-uri de bază, specificată și ea ca argument.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg)

Acum, obiectele `YGate` sunt absente din Circuit-ul nostru și există informații explicite de sincronizare sub forma Gate-urilor `Delay`. Acest Circuit transpilat cu decuplare dinamică este acum gata să fie trimis la Backend.

## Pașii următori

> **Tip:** - Pentru a afla cum să folosești funcția `generate_preset_passmanager` în loc să îți scrii propriile pass-uri, începe cu subiectul [Setările implicite de transpilare și opțiunile de configurare](defaults-and-configuration-options).
> - Încearcă ghidul [Compară setările Transpiler-ului](/guides/circuit-transpilation-settings#compare-transpiler-settings).
> - Consultă [documentația API pentru Transpile](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler).